# Systematic uncertainty breakdown (final selected variables)

This notebook loads the precomputed systematic covariance files written by the workflow drivers:

- `run_syst_cosmics_chunked.sh` -> `Cosmics/cosmics_syst_dict.npz`
- `run_syst_multisim_chunked.sh` -> `MCstat/`, `Flux/`, `G4/`
- `run_syst_detvar_chunked.sh` -> `Detector/detector_syst_dict.npz`
- `run_syst_genie_chunked.sh` (seeded/produced elsewhere) -> `GENIE/cov_mat_dict.pkl`

It then computes, **for each final selected variable**:

1. A **summary** plot of the integrated-rate uncertainty contribution from each source (Flux, G4, detvar, cosmic, GENIE) and the quadrature total.
2. For **each source**, the **top 5 bin drivers** (largest contribution to the integrated-rate variance), shown in a per-source bar plot.

## Notes on the decomposition
The integrated-rate fractional variance for a variable with `n` bins is:

V = 1^T C 1

where `C` is the source fractional covariance matrix and `1` is an `n`-vector of ones.

For ranking “top drivers” we compute the per-bin signed variance contributions:

t_i = (C * 1)_i

and rank by `|t_i|`.


In [ ]:
import os
from pathlib import Path
import pickle
import numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.append('/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana') # absolute path for running on EAF
from analysis_village.numucc_1p0pi.dataset_locations import default_syst_disk_root, PLOTS_BASE
from analysis_village.numucc_1p0pi.final_selected_evt_vars import (
    CORE_SELECTED_EVT_VARIABLE_CONFIGS,
    with_final_selected_evt_variables,
)



ModuleNotFoundError: No module named 'analysis_village.numucc_1p0pi.dataset_locations'

In [ ]:
# ----------------------------
# User configuration
# ----------------------------

# Root of the syst_disk_layout tree. Must contain:
#   MCstat/mcstat_syst_dict.npz
#   Flux/flux_syst_dict.npz
#   G4/g4_syst_dict.npz
#   Cosmics/cosmics_syst_dict.npz
#   Detector/detector_syst_dict.npz
#   GENIE/cov_mat_dict.pkl
SYST_DISK_ROOT = os.environ.get('NUMUCC_SYST_DISK_ROOT', None)
if not SYST_DISK_ROOT:
    SYST_DISK_ROOT = str(default_syst_disk_root())
SYST_DISK_ROOT = str(Path(SYST_DISK_ROOT).expanduser())

# Subset of variables to plot (None = all final selected)
# Provide strings like 'integrated', 'muon-p', etc.
VARS_TO_PLOT = None

# Output directory for figures
OUT_DIR = Path(PLOTS_BASE) / 'syst_uncertainty_breakdown' / 'final_selected'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Sources to include
SOURCES = ['Flux', 'G4', 'detvar', 'cosmic', 'GENIE']

# Top-k bin drivers per source
TOP_K = 5

print('SYST_DISK_ROOT =', SYST_DISK_ROOT)
print('OUT_DIR       =', str(OUT_DIR))


In [ ]:
# ----------------------------
# Variable catalogue
# ----------------------------

var_configs = with_final_selected_evt_variables(list(CORE_SELECTED_EVT_VARIABLE_CONFIGS))

if VARS_TO_PLOT is not None:
    allowed = set(VARS_TO_PLOT)
    var_configs = [vc for vc in var_configs if vc.var_save_name in allowed]

var_names = [vc.var_save_name for vc in var_configs]
print(f'Final selected variables: {len(var_names)}')
print(' - ' + ', '.join(var_names))


In [ ]:
# ----------------------------
# Load covariance payloads
# ----------------------------

from analysis_village.numucc_1p0pi.syst_disk_layout import syst_disk_paths

paths = syst_disk_paths(SYST_DISK_ROOT)

needed = {
    'flux_npz': paths['flux'],
    'g4_npz': paths['g4'],
    'cosmics_npz': paths['cosmics'],
    'detector_npz': paths['detector'],
    'genie_pkl': paths['genie'],
}

missing = [k for k,p in needed.items() if not Path(p).is_file()]
if missing:
    raise FileNotFoundError(
        'Missing required syst files:
' + '
'.join(f'  {k}: {needed[k]}' for k in missing)
    )

flux_npz = np.load(needed['flux_npz'], allow_pickle=True)
g4_npz = np.load(needed['g4_npz'], allow_pickle=True)
cosmics_npz = np.load(needed['cosmics_npz'], allow_pickle=True)
detector_npz = np.load(needed['detector_npz'], allow_pickle=True)

with open(needed['genie_pkl'], 'rb') as gf:
    genie_blob = pickle.load(gf)

print('Loaded syst covariance payloads.')


In [ ]:
def _get_cov_frac_from_legacy_npz(npz_obj, var_save_name: str, inner_key: str):
    # Matches logic in analysis_village.numucc_1p0pi.utils.get_syst_unc
    entry = dict(npz_obj)[var_save_name].item()
    return np.asarray(entry[inner_key]['cov_frac'], dtype=float)


def _get_cov_frac_detvar(detector_npz_obj, var_save_name: str):
    # Matches logic in analysis_village.numucc_1p0pi.utils.get_syst_unc
    return np.asarray(dict(detector_npz_obj)['detector'].item()[var_save_name]['cov_frac'], dtype=float)


def _get_cov_frac_genie(genie_blob_obj, var_save_name: str):
    g = genie_blob_obj[var_save_name]['genie']
    if isinstance(g, dict):
        if 'cov_frac' in g:
            return np.asarray(g['cov_frac'], dtype=float)
        raise ValueError(f"GENIE payload dict lacks 'cov_frac' for {var_save_name}; keys={list(g.keys())}")
    return np.asarray(g, dtype=float)


def _integrated_sd_percent_from_cov_frac(cov_frac: np.ndarray):
    n = cov_frac.shape[0]
    ones = np.ones(n, dtype=float)
    var = float(ones @ cov_frac @ ones)
    var = max(var, 0.0)  # numerical safety
    return 100.0 * np.sqrt(var)


def _topk_bin_drivers_from_cov_frac(cov_frac: np.ndarray, bin_edges: np.ndarray, top_k: int):
    n = cov_frac.shape[0]
    ones = np.ones(n, dtype=float)
    contrib = cov_frac @ ones  # per-bin signed variance contributions (fraction^2)

    idx = np.argsort(np.abs(contrib))[::-1][: min(top_k, n)]

    labels = []
    for i in idx:
        lo = float(bin_edges[i])
        hi = float(bin_edges[i+1])
        labels.append(f'{lo:g}-{hi:g}')

    # Effective “uncertainty contribution” scale: sqrt(|variance contribution|)
    eff_sd_percent = 100.0 * np.sqrt(np.abs(contrib[idx]))

    return idx, labels, contrib[idx], eff_sd_percent


def get_source_cov_frac(var_save_name: str):
    covs = {}
    covs['Flux'] = _get_cov_frac_from_legacy_npz(flux_npz, var_save_name, inner_key='flux')
    covs['G4'] = _get_cov_frac_from_legacy_npz(g4_npz, var_save_name, inner_key='G4')
    covs['cosmic'] = _get_cov_frac_from_legacy_npz(cosmics_npz, var_save_name, inner_key='Cosmics')
    covs['detvar'] = _get_cov_frac_detvar(detector_npz, var_save_name)
    covs['GENIE'] = _get_cov_frac_genie(genie_blob, var_save_name)
    return covs

print('Defined covariance loader + decomposition helpers.')


In [ ]:
def plot_one_variable(var_config):
    vn = var_config.var_save_name
    bins = np.asarray(var_config.bins, dtype=float)

    covs = get_source_cov_frac(vn)

    # Integrated SD per source
    sd_by_source = {src: _integrated_sd_percent_from_cov_frac(covs[src]) for src in SOURCES}

    # Total (quadrature) from independent source covariances
    var_total = 0.0
    for src in SOURCES:
        n = covs[src].shape[0]
        ones = np.ones(n, dtype=float)
        var_src = float(ones @ covs[src] @ ones)
        var_total += max(var_src, 0.0)
    sd_total = 100.0 * np.sqrt(max(var_total, 0.0))

    # ----------------------------
    # Summary plot
    # ----------------------------
    src_order = ['Flux', 'G4', 'detvar', 'cosmic', 'GENIE']
    labels = src_order + ['Total']
    values = [sd_by_source[s] for s in src_order] + [sd_total]

    fig, ax = plt.subplots(figsize=(9.2, 4.9))
    x = np.arange(len(labels))
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', 'black']
    ax.bar(x, values, color=colors, edgecolor='none')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=25, ha='right')
    ax.set_ylabel('Integrated-rate frac. uncertainty [%]')
    ax.set_title(vn)
    ax.grid(axis='y', alpha=0.25)
    fig.tight_layout()

    out_summary = OUT_DIR / f'{vn}__summary_sources.png'
    fig.savefig(out_summary, dpi=160)
    plt.show()
    plt.close(fig)

    # ----------------------------
    # Top-k drivers per source
    # ----------------------------
    fig2, axes = plt.subplots(1, 5, figsize=(17.5, 4.8), sharey=False)
    fig2.suptitle(f'{vn} — top-{TOP_K} bin drivers by |variance contrib|', y=1.02)

    for ax, src in zip(axes, src_order):
        idx, labels_bin, contrib_signed, eff_sd = _topk_bin_drivers_from_cov_frac(
            covs[src], bins, TOP_K
        )

        ax.bar(np.arange(len(labels_bin)), eff_sd, color='#4c78a8', edgecolor='none')
        ax.set_xticks(np.arange(len(labels_bin)))
        ax.set_xticklabels(labels_bin, rotation=45, ha='right', fontsize=8)
        ax.set_title(src)
        ax.set_ylabel('Effective contrib [%, sqrt(|var|)]')
        ax.grid(axis='y', alpha=0.25)

    fig2.tight_layout()
    out_top = OUT_DIR / f'{vn}__topk_drivers_by_source.png'
    fig2.savefig(out_top, dpi=160, bbox_inches='tight')
    plt.show()
    plt.close(fig2)

    top_table = {}
    for src in src_order:
        _, labels_bin, contrib_signed, eff_sd = _topk_bin_drivers_from_cov_frac(
            covs[src], bins, TOP_K
        )
        top_table[src] = {
            'bin_labels': labels_bin,
            'var_contrib_fraction2': [float(x) for x in contrib_signed],
            'eff_sd_percent': [float(x) for x in eff_sd],
        }

    return {
        'var_save_name': vn,
        'sd_by_source_percent': sd_by_source,
        'sd_total_percent': float(sd_total),
        'top_table': top_table,
        'figures': {
            'summary': str(out_summary),
            'topk': str(out_top),
        },
    }

print('Ready to plot variables.')


In [ ]:
results = []
for i, vc in enumerate(var_configs, 1):
    print(f'[{i}/{len(var_configs)}] {vc.var_save_name}')
    res = plot_one_variable(vc)
    results.append(res)

print('Done. Produced', len(results), 'variables.')


In [ ]:
# Optional: tabulate integrated-rate uncertainties across sources
import pandas as pd

rows = []
for res in results:
    vn = res['var_save_name']
    row = {'var_save_name': vn, 'sd_total_percent': res['sd_total_percent']}
    row.update({f'sd_{k}_percent': v for k,v in res['sd_by_source_percent'].items()})
    rows.append(row)

df = pd.DataFrame(rows).sort_values('var_save_name')
print(df.to_string(index=False))

out_csv = OUT_DIR / 'integrated_uncertainties_by_source.csv'
df.to_csv(out_csv, index=False)
print('Wrote', str(out_csv))
